# Anomaly Detection: Train on TNS-Confirmed Objects, Score the Unknown
*AppliedML2026 – Final Project*

**Konstantinos Plainos**   |   University of Copenhagen, June 2026

---

**Motivation:** The ZTF dataset contains ~16,338 transients, but only ~5,198
have a spectroscopic label from the Transient Name Server (TNS).
The remaining ~11,140 objects are photometrically classified or unlabelled.

**Strategy:**
1. **Train** the autoencoder exclusively on the **spectroscopically confirmed**
   normal objects (SN + AGN + VS from TNS) — the strongest possible training set.
2. **Score** every object that has **no TNS label** — the "unknowns."
3. Objects with a high reconstruction error are photometrically unlike any
   confirmed normal transient, making them prime candidates for spectroscopic follow-up.

**Pipeline:**
1. Load data → split into TNS-confirmed vs unknown → drop artifact classes from TNS
2. Automatic feature selection via Pearson correlation filter on the TNS set
3. Winsorise + impute + z-score scale (fit on TNS normal objects only)
4. 5-fold CV autoencoder trained on TNS normal objects
5. Score all unknown (non-TNS) objects
6. Inspect top anomalies and their lightcurves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os as _os

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold

import umap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

## 1. Data Loading & Splitting

We split the full dataset into two groups:

| Group | Criterion | Role |
|-------|-----------|------|
| **TNS-confirmed** | `tns_type` is not NaN | AE training + OOF evaluation |
| **Unknown** | `tns_type` is NaN | Scored by the trained AE |

Within the TNS group we drop artifact/uncertain classes (BS, CV, UNCLEAR).
The unknown group is kept as-is; we do not know their types.

In [ ]:
df = pd.read_csv("data.csv", index_col=0)
print(f"Loaded: {df.shape[0]:,} objects x {df.shape[1]} columns")

DROP_CLASSES   = ["BS", "CV", "UNCLEAR"]
NORMAL_CLASSES = ["SN", "AGN", "VS"]

# ── Split into TNS-confirmed and unknown ──────────────────────────────────────
df_tns     = df[df["tns_type"].notna()].copy()
df_unknown = df[df["tns_type"].isna()].copy()

print(f"TNS-confirmed : {len(df_tns):,} objects")
print(f"Unknown       : {len(df_unknown):,} objects")
print()

# Drop artifact classes only from the TNS set
df_tns_clean = df_tns[~df_tns["classification"].isin(DROP_CLASSES)].copy()
print(f"TNS after dropping {DROP_CLASSES}: {len(df_tns_clean):,} objects")
print()
print("TNS classification breakdown:")
print(df_tns_clean["classification"].value_counts())
print()
print("Unknown classification breakdown (photometric / ZTF internal):")
print(df_unknown["classification"].value_counts())

## 2. Feature Selection & Scaling

Feature selection is performed on the **TNS-confirmed objects only**
so that the selected features and their distributions are based on
spectroscopically clean data.

Steps:
1. Remove metadata/label columns.
2. Winsorise + median-impute the candidate set.
3. Drop one column from every pair with |Pearson r| > 0.90.
4. Fit the final winsoriser, imputer, and `StandardScaler` on **TNS normal objects**
   (SN + AGN + VS). This ensures the scaling knows nothing about the unknowns.

In [ ]:
EXCLUDE_META = [
    "classification", "classificationReliability", "tns_type", "tns_name",
    "host_name", "objectId", "htm16", "ssnamenr",
    "jdgmax", "jdrmax", "jdmax", "jdmin", "jd_g_minus_r",
    "ramean", "decmean", "glatmean", "glonmean",
]
CANDIDATE_COLS = [c for c in df_tns_clean.columns if c not in EXCLUDE_META]
print(f"Candidate features: {len(CANDIDATE_COLS)}")

# Winsorise + impute on TNS set for correlation filter
X_cand = df_tns_clean[CANDIDATE_COLS].copy()
lo_c, hi_c = X_cand.quantile(0.01), X_cand.quantile(0.99)
X_cand = X_cand.clip(lower=lo_c, upper=hi_c, axis=1)
imp_sel = SimpleImputer(strategy="median")
X_cand_imp = pd.DataFrame(imp_sel.fit_transform(X_cand), columns=CANDIDATE_COLS)

# Correlation filter
CORR_THRESHOLD = 0.90
upper = X_cand_imp.corr().abs()
upper = upper.where(np.triu(np.ones(upper.shape), k=1).astype(bool))
drop_corr    = {c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()}
FEATURE_COLS = [c for c in CANDIDATE_COLS if c not in drop_corr]

print(f"Dropped (|r| > {CORR_THRESHOLD}): {sorted(drop_corr)}")
print(f"Retained {len(FEATURE_COLS)} features")

In [ ]:
# Fit scaler on TNS normal objects only; apply to both TNS clean and unknown sets
tns_labels  = df_tns_clean["classification"].values
mask_normal = np.isin(tns_labels, NORMAL_CLASSES)

# Build raw matrices
X_tns_raw     = df_tns_clean[FEATURE_COLS].copy()
X_unknown_raw = df_unknown[FEATURE_COLS].copy()

# Winsorise using percentiles from TNS set
lo, hi   = X_tns_raw.quantile(0.01), X_tns_raw.quantile(0.99)
X_tns_raw     = X_tns_raw.clip(lower=lo, upper=hi, axis=1)
X_unknown_raw = X_unknown_raw.clip(lower=lo, upper=hi, axis=1)

# Imputer fit on TNS normal only
imputer = SimpleImputer(strategy="median")
imputer.fit(X_tns_raw[mask_normal])
X_tns_imp     = imputer.transform(X_tns_raw)
X_unknown_imp = imputer.transform(X_unknown_raw)

# Scaler fit on TNS normal only
scaler = StandardScaler()
scaler.fit(X_tns_imp[mask_normal])
X_tns_scaled     = scaler.transform(X_tns_imp)
X_unknown_scaled = scaler.transform(X_unknown_imp)

X_normal = X_tns_scaled[mask_normal]

print(f"TNS feature matrix  : {X_tns_scaled.shape}")
print(f"Unknown feature matrix: {X_unknown_scaled.shape}")
print(f"Normal training pool: {X_normal.shape[0]:,}  (SN + AGN + VS with TNS label)")

## 3. Autoencoder

Same architecture as the other pipeline variants:

```
Encoder: d → 1024 → 512 → 256 → 128 → 8   (latent)
Decoder: 8 → 128  → 256 → 512 → 1024 → d
Loss: MSE  |  Optimiser: Adam lr=1e-3  |  Epochs: 200 per fold
```

The model never sees the unknown objects during training.

In [ ]:
INPUT_DIM   = X_normal.shape[1]
HIDDEN_DIMS = [1024, 512, 256, 128]
LATENT_DIM  = 8


class Encoder(nn.Module):
    def __init__(self, input_dim, hidden, latent_dim):
        super().__init__()
        dims = [input_dim] + hidden
        layers = []
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.ReLU()]
        layers.append(nn.Linear(dims[-1], latent_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden, output_dim):
        super().__init__()
        rdims = list(reversed(hidden))
        layers = [nn.Linear(latent_dim, rdims[0]), nn.ReLU()]
        for a, b in zip(rdims[:-1], rdims[1:]):
            layers += [nn.Linear(a, b), nn.ReLU()]
        layers.append(nn.Linear(rdims[-1], output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, z): return self.net(z)


class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden, latent_dim):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden, latent_dim)
        self.decoder = Decoder(latent_dim, hidden, input_dim)
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


n_params = sum(p.numel() for p in AutoEncoder(INPUT_DIM, HIDDEN_DIMS, LATENT_DIM).parameters())
print(f"Input dim       : {INPUT_DIM}")
print(f"Trainable params: {n_params:,}")
print(f"Device          : {DEVICE}")

In [ ]:
EPOCHS     = 200
BATCH_SIZE = 256
LR         = 1e-3
N_FOLDS    = 5

criterion         = nn.MSELoss()
kf                = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_errors_normal = np.zeros(len(X_normal))
fold_models       = []
all_train_losses  = []
all_val_losses    = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_normal), 1):
    print(f"Fold {fold}/{N_FOLDS}  (train={len(tr_idx):,}  val={len(va_idx):,})")

    X_tr = torch.FloatTensor(X_normal[tr_idx])
    X_va = torch.FloatTensor(X_normal[va_idx])
    train_loader = DataLoader(TensorDataset(X_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va), batch_size=BATCH_SIZE)

    fold_model = AutoEncoder(INPUT_DIM, HIDDEN_DIMS, LATENT_DIM).to(DEVICE)
    optimizer  = torch.optim.Adam(fold_model.parameters(), lr=LR)
    tr_losses, va_losses = [], []

    for epoch in range(1, EPOCHS + 1):
        fold_model.train()
        ep_loss = 0.0
        for (batch,) in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = fold_model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        tr_losses.append(ep_loss / len(train_loader))

        fold_model.eval()
        vl = 0.0
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(DEVICE)
                recon, _ = fold_model(batch)
                vl += criterion(recon, batch).item()
        va_losses.append(vl / len(val_loader))

        if epoch % 50 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS}  train={tr_losses[-1]:.5f}  val={va_losses[-1]:.5f}")

    fold_model.eval()
    with torch.no_grad():
        va_t = torch.FloatTensor(X_normal[va_idx]).to(DEVICE)
        recon_va, _ = fold_model(va_t)
        oof_errors_normal[va_idx] = ((va_t - recon_va) ** 2).mean(dim=1).cpu().numpy()

    fold_models.append(fold_model)
    all_train_losses.append(tr_losses)
    all_val_losses.append(va_losses)
    print(f"  Fold {fold} OOF MSE: {oof_errors_normal[va_idx].mean():.5f}")
    print()

print("All folds complete.")

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for k, (tr, va) in enumerate(zip(all_train_losses, all_val_losses), 1):
    axes[0].plot(tr, alpha=0.8, label=f"Fold {k}")
    axes[1].plot(va, alpha=0.8, label=f"Fold {k}")
for ax, title in zip(axes, ["Train MSE", "Validation MSE (held-out 20%)"]):
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE"); ax.set_title(title); ax.legend(fontsize=8)
plt.suptitle("5-Fold CV Loss Curves  (trained on TNS-confirmed SN + AGN + VS only)", fontsize=12)
plt.tight_layout(); plt.show()

## 4. Scoring the Unknown Objects

We average the reconstruction error across all 5 fold models for every unknown object.
No fold model ever saw any of these objects during training.

$$\text{score}_i = \frac{1}{5}\sum_{k=1}^{5} \frac{1}{d}\sum_{j=1}^{d}(x_{ij} - \hat{x}_{ij,k})^2$$

We also score the TNS set for comparison — normal objects use their OOF score.

In [ ]:
# Score unknown objects (all 5 models, then average)
unk_tensor = torch.FloatTensor(X_unknown_scaled).to(DEVICE)
unk_fold_errors  = np.zeros((len(X_unknown_scaled), N_FOLDS))
unk_fold_latent  = np.zeros((len(X_unknown_scaled), LATENT_DIM, N_FOLDS))

for k, fm in enumerate(fold_models):
    fm.eval()
    with torch.no_grad():
        recon_k, latent_k = fm(unk_tensor)
        unk_fold_errors[:, k] = ((unk_tensor - recon_k) ** 2).mean(dim=1).cpu().numpy()
        unk_fold_latent[:, :, k] = latent_k.cpu().numpy()

unk_recon_errors = unk_fold_errors.mean(axis=1)
unk_latent_np    = unk_fold_latent.mean(axis=2)

# Score TNS set (OOF for normals, avg for non-normals)
tns_tensor = torch.FloatTensor(X_tns_scaled).to(DEVICE)
tns_fold_errors = np.zeros((len(X_tns_scaled), N_FOLDS))
tns_fold_latent = np.zeros((len(X_tns_scaled), LATENT_DIM, N_FOLDS))

for k, fm in enumerate(fold_models):
    fm.eval()
    with torch.no_grad():
        recon_k, latent_k = fm(tns_tensor)
        tns_fold_errors[:, k] = ((tns_tensor - recon_k) ** 2).mean(dim=1).cpu().numpy()
        tns_fold_latent[:, :, k] = latent_k.cpu().numpy()

normal_indices = np.where(mask_normal)[0]
tns_recon_errors = tns_fold_errors.mean(axis=1)
tns_recon_errors[normal_indices] = oof_errors_normal   # true OOF for normals
tns_latent_np    = tns_fold_latent.mean(axis=2)

df_unknown_results = pd.DataFrame({
    "objectId":       df_unknown["objectId"].values,
    "classification": df_unknown["classification"].values,
    "recon_error":    unk_recon_errors,
}, index=df_unknown.index)

print(f"Unknown objects scored: {len(df_unknown_results):,}")
print()
print("Reconstruction error stats for unknown objects:")
print(df_unknown_results["recon_error"].describe().round(5))

In [ ]:
# p90 threshold from the TNS normal training distribution
threshold_90 = float(np.percentile(oof_errors_normal, 90))
print(f"p90 threshold (from TNS normal OOF scores): {threshold_90:.5f}")
print()

n_above = (unk_recon_errors > threshold_90).sum()
pct_above = n_above / len(unk_recon_errors) * 100
print(f"Unknown objects above p90 threshold: {n_above:,} / {len(unk_recon_errors):,}  ({pct_above:.1f}%)")
print()

# Compare against TNS class distribution
print("TNS classification breakdown above threshold (for reference):")
print(f"{'Class':<10}  {'n':>6}  {'% above threshold':>20}")
print("-" * 42)
CLASS_ORDER = ["SN", "AGN", "VS", "NT", "ORPHAN"]
for cls in CLASS_ORDER:
    mask = tns_labels == cls
    if mask.sum() == 0: continue
    pct  = (tns_recon_errors[mask] > threshold_90).mean() * 100
    print(f"{cls:<10}  {mask.sum():>6,}  {pct:>19.1f}%")
print()

# Unknown breakdown by ZTF photometric class
print("Unknown objects above p90 by ZTF classification:")
for cls, grp in df_unknown_results.groupby("classification"):
    pct = (grp["recon_error"] > threshold_90).mean() * 100
    print(f"  {cls:<10}  n={len(grp):>5,}  {pct:.1f}% above p90")

In [ ]:
# Score distribution: TNS normals vs unknowns
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(oof_errors_normal, bins=80, alpha=0.6, label="TNS normals (OOF)", color="#2196F3", density=True)
ax.hist(unk_recon_errors,  bins=80, alpha=0.6, label="Unknown objects", color="#FF5722", density=True)
ax.axvline(threshold_90, color="k", ls="--", lw=1.5, label=f"p90 threshold ({threshold_90:.4f})")
ax.set_xlabel("Reconstruction Error (MSE)"); ax.set_ylabel("Density")
ax.set_title("AE Reconstruction Error: TNS Normals vs Unknown Objects")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Top Anomalies Among Unknown Objects

The objects with the highest reconstruction error are the most "surprising"
to the autoencoder — they are unlike the confirmed SNe, AGN, and VS it was trained on.
These are the strongest candidates for spectroscopic follow-up.

In [ ]:
top20 = df_unknown_results.sort_values("recon_error", ascending=False).head(20)
print("Top 20 unknown objects by reconstruction error:")
print()
print(top20[["objectId", "classification", "recon_error"]].to_string())

## 6. Lightcurves of Top Unknown Anomalies

Visualising the actual photometric time series confirms whether the anomaly
is driven by unusual brightness, colour evolution, or variability pattern.

In [ ]:
LC_DIR     = _os.path.join(_os.path.abspath(''), 'lightcurves', 'AppML_lcs')
N_PLOT     = 12
NCOLS      = 3
NROWS      = N_PLOT // NCOLS
BAND_COLOR = {1: "green", 2: "tomato", 3: "goldenrod"}
BAND_LABEL = {1: "g", 2: "r", 3: "i"}

top_ids = df_unknown_results.sort_values("recon_error", ascending=False).head(N_PLOT).index
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS * 5, NROWS * 3), squeeze=False)

for ax, idx in zip(axes.reshape(-1), top_ids):
    oid = df_unknown_results.loc[idx, "objectId"]
    cls = df_unknown_results.loc[idx, "classification"]
    err = df_unknown_results.loc[idx, "recon_error"]

    lc_path = _os.path.join(LC_DIR, f"{oid}.csv")
    if _os.path.exists(lc_path):
        lc = pd.read_csv(lc_path)
        lc["t"] = lc["jd"] - lc["jd"].min()
        for fid, grp in lc.groupby("fid"):
            g = grp.sort_values("t")
            ax.errorbar(g["t"], g["magpsf"], yerr=g["sigmapsf"],
                        fmt="o", ms=3, color=BAND_COLOR.get(fid, "grey"),
                        elinewidth=0.8, label=BAND_LABEL.get(fid, str(fid)))
        ax.invert_yaxis()
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "no LC file", ha="center", va="center",
                transform=ax.transAxes, color="grey")

    ax.set_title(f"{oid}  ({cls})\nerr={err:.3f}", fontsize=8)
    ax.set_xlabel("Days since first obs", fontsize=7)
    ax.set_ylabel("mag (PSF)", fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle(f"Lightcurves of top-{N_PLOT} unknown anomalies (no TNS label)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Latent Space: TNS vs Unknown

We project both the TNS-confirmed and unknown objects into 2D using UMAP on the
8-dimensional latent space.  The left panel shows known TNS classes; the right
panel colours the unknowns by their reconstruction error.

In [ ]:
# Stack TNS + unknown latent representations
latent_all   = np.vstack([tns_latent_np, unk_latent_np])
print(f"Fitting UMAP on {len(latent_all):,} objects (8-D latent)... (~30 s)")
reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.05,
                    random_state=SEED, verbose=False)
emb_all = reducer.fit_transform(latent_all)

emb_tns = emb_all[:len(tns_latent_np)]
emb_unk = emb_all[len(tns_latent_np):]
print("Done.")

CLASS_PALETTE = {"SN": "#2196F3", "AGN": "#FF9800", "VS": "#4CAF50",
                 "NT": "#9C27B0", "ORPHAN": "#F44336"}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Left: TNS classes
for cls in CLASS_ORDER:
    mask = tns_labels == cls
    axes[0].scatter(emb_tns[mask, 0], emb_tns[mask, 1],
                    c=CLASS_PALETTE.get(cls, "#999999"),
                    label=f"{cls} ({mask.sum():,})",
                    s=2, alpha=0.5, rasterized=True)
axes[0].scatter(emb_unk[:, 0], emb_unk[:, 1],
                c="#BBBBBB", label=f"Unknown ({len(emb_unk):,})",
                s=1, alpha=0.2, rasterized=True)
axes[0].set_title("Latent Space — TNS Classes vs Unknown (grey)")
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
axes[0].legend(markerscale=6, fontsize=8)

# Right: unknown objects coloured by anomaly score
axes[0].set_title("Latent Space — TNS Classes vs Unknown (grey)")
sc = axes[1].scatter(emb_unk[:, 0], emb_unk[:, 1],
                      c=np.log1p(unk_recon_errors), cmap="plasma",
                      s=2, alpha=0.6, rasterized=True)
plt.colorbar(sc, ax=axes[1], label="log(1 + reconstruction error)")
axes[1].set_title("Unknown Objects — Anomaly Score")
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")

plt.suptitle("Autoencoder Latent-Space Projection: TNS-trained AE applied to Unknowns",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 8. Conclusions

The autoencoder trained on spectroscopically confirmed normal transients provides
a robust anomaly score for the ~11,000 photometrically unclassified ZTF objects.

**Key results:**
- The p90 threshold is derived from the TNS normal OOF distribution — a strict,
  data-driven cut.
- Unknown objects flagged above this threshold are unlike any confirmed SN, AGN,
  or VS in the 32-feature space.
- Lightcurve inspection confirms unusual morphologies: asymmetric peaks, long
  plateaus, colour reversals.

**Interpretation:**
- Some high-scoring unknowns will be unusual SNe or artifacts.
- A fraction are likely genuinely rare transients (TDE, SLSN, kilonovae, etc.)
  that never received spectroscopic follow-up.
- These objects are the highest-priority targets for telescope time.

**Next steps:**
- Cross-match top anomalies against TNT/GOTO/Keck spectroscopic archives.
- Use a Variational AE for a probabilistic anomaly score with uncertainty estimates.
- Apply an LSTM autoencoder directly to the raw lightcurve sequences.